In [ ]:
!pip install torch
!pip install torch_geometric
!pip install rdkit
!pip install torcheval
!pip install matplotlib
!git clone https://github.com/olsson-group/sml_project1.git

In [ ]:
from sml_project1 import metrics, data
import torch
import torch_geometric as geom
from torch_geometric.datasets import MoleculeNet
import matplotlib.pyplot as plt

In [ ]:
class MolecularGraphNet(torch.nn.Module):
  # This is the parent class for both the regression and classification task

  def __init__(self, embedding, net):
    super().__init__()
    self.embedding = embedding
    self.net = net
    self.setup_optimizer()

  def training_step(self, loss):
    loss.backward()
    self.optimizer.step()
    self.optimizer.zero_grad()

  def setup_optimizer(self):
    self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)

  def forward(self, batch):
    # Implement in children
    raise NotImplementedError

  def loss(self, y, y_hat):
    # Implement in children
    raise NotImplementedError


class MolecularGraphClassification(MolecularGraphNet):
  def __init__(self, embedding, net):
    super().__init__(embedding, net)

  def forward(self, batch):
    x = batch.x.to(torch.float32)  # prevent a type mismatch

    # TODO: Setup the forward pass of the model for a classification task, this
    # includes embedding the input data, applying the graph network, pooling
    # the results, and predicting an output.
    # HINT: Pooling can be done using
    #     x = geom.nn.global_mean_pool(batch.x, batch.batch)
    # where batch.x denotes the node features, and batch.batch denotes which
    # graph each node belongs to

    raise NotImplementedError

  def loss(self, y, y_hat):
    # TODO: Setup the appropriate loss for training a classification model.
    raise NotImplementedError

  def evaluate(self, y, y_hat):
    roc_auc = metrics.get_roc_auc(y, y_hat)
    return roc_auc

class MolecularGraphRegression(MolecularGraphNet):
  def __init__(self, embedding, net):
    super().__init__(embedding, net)

  def forward(self, batch):
    # TODO: Setup the forward pass of the model for a classification task, this
    # includes embedding the input data, applying the graph network, pooling
    # the results, and predicting an output.
    x = batch.x.to(torch.float32)  # prevent a type mismatch

    raise NotImplementedError


  def loss(self, y, y_hat):
    # TODO: Setup the appropriate loss for training a regression model.
    return torch.nn.functional.mse_loss(y, y_hat)


  def evaluate(self, y, y_hat):
    pcc = metrics.get_pearson_corr(y, y_hat)
    rmse = metrics.get_rmse(y, y_hat)
    return pcc, rmse

In [ ]:
class DeepSet(torch.nn.Module):
  # TODO: Implement DeepSet model.
  def __init__(self, in_channels, hidden_channels, out_channels):
    super().__init__()
    raise NotImplementedError

  def forward(self, x, edge_index, batch):
    raise NotImplementedError

In [ ]:
class GCN(torch.nn.Module):
  # TODO: Implement GCN model.
  def __init__(self, in_channels, hidden_channels, out_channels):
    super().__init__()
    raise NotImplementedError

  def forward(self, x, edge_index, batch):
    raise NotImplementedError


In [ ]:
class GAT(torch.nn.Module):
  # TODO: Implement GAT model.
  def __init__(self, in_channels, hidden_channels, out_channels):
    super().__init__()
    raise NotImplementedError


  def forward(self, x, edge_index, batch):
    raise NotImplementedError


In [ ]:
# REGRESSION TASK: DATA

# Download ESOL data, filter small molecules and split into training and
# validation dataset
esol = data.filter_dataset(MoleculeNet(root='./storage/', name='ESOL'))
train_dataset, validation_dataset = data.split_dataset(esol, frac_train = .8, seed=None, split=None)

in_features = 9 # There are 9 features in the ESOL dataset

AssertionError: please provide a seed for the splitting for reproducibility

In [ ]:
# sol# REGRESSION TASK: DATA

# Download ESOL data, filter small molecules and split into training and
# validation dataset
esol = data.filter_dataset(MoleculeNet(root='./storage/', name='ESOL'))
train_dataset, validation_dataset = data.split_dataset(esol, frac_train = .8, seed=42, split='random')

in_features = 9 # There are 9 features in the ESOL dataset

In [ ]:
# REGRESSION TASK: MODELS

in_features = ...
hidden_features = ...
out_features = ...

# Embedding
# TODO: implement an embedding for your models
embedding = ...

deepset_regression = MolecularGraphRegression(
    embedding,
    DeepSet(in_features, hidden_features, out_features),
)

gcn_regression = MolecularGraphRegression(
    embedding,
    GCN(in_features, hidden_features, out_features),
)

gat_regression = MolecularGraphRegression(
    embedding,
    GAT(in_features, hidden_features, out_features),
)

In [ ]:
# REGRESSION TASK: TRAINING

train_batch_size = ...
val_batch_size = ...
n_epochs = ...

train_dataloader = geom.loader.DataLoader(
    train_dataset, batch_size=train_batch_size, shuffle=True
)

val_dataloader = geom.loader.DataLoader(
    val_dataset, batch_size=val_batch_size
)

model = ...

# EXAMPLE of training script
for epoch in range(n_epochs):
  for batch in train_dataloader:
    y_hat = model.forward(batch)
    train_loss = model.loss_fn(batch.y, y_hat)
    train_metrics = model.evaluate(batch.y, y_hat)
    model.training_step(train_loss)

    print(train_loss)

  for val_batch in val_dataloader:
    y_hat = model.forward(val_batch)
    val_loss = model.loss_fn(val_batch.y, y_hat)
    val_metrics = model.evaluate(val_batch)



ValueError: batch_size should be a positive integer value, but got batch_size=Ellipsis

In [ ]:
# TODO: Present results for the regression task and compare model types. This
# can be in the form of tables or graphs or however you see fit

In [ ]:
# CLASSIFICATION TASK: DATA

bbbp = data.filter_dataset(MoleculeNet(root='./storage/', name='BBBP'))
train_dataset, val_dataset = data.split_dataset(esol, frac_train = .8)

in_features = 9 # There are 9 features in the BBBP dataset

In [ ]:
# CLASSIFICATION TASK: MODELS

# Embedding
# TODO: implement an embedding for your models
embedding = ...

# Regression models
deepset_classification = MolecularGraphClassification(
    embedding,
    DeepSet(hidden_features, hidden_features, 1),
)

gcn_classification = MolecularGraphClassification(
    embedding,
    GCN(hidden_features, hidden_features, 1),
)

gat_classification = MolecularGraphClassification(
    embedding,
    GAT(hidden_features, hidden_features, 1),
)

In [ ]:
# CLASSIFICATION TASK: TRAINING
train_batch_size = ...
val_batch_size = ...
n_epochs = ...

train_dataloader = geom.loader.DataLoader(
  train_dataset, batch_size=batch_size, shuffle=True
)
val_dataloader = geom.loader.DataLoader(
    val_dataset, batch_size=val_batch_size)

model = ...

# TODO: Again, train 3 models of each type and evaluate their loss and metrics

In [ ]:
# Present results for the classification task.
# This can be in the form of tables or graphs.